# Does TGLF's GHAT $\approx 0.5$ drift-frequency fix generalise from `sample_0818` to the 738 KBMs? (`Fusion_PhD-qp5.48`)

On the reference KBM (SPR-045 `sample_0818`) TGLF's drift-frequency coefficient GHAT, $\omega_{d1}=-\mathrm{GHAT}\,k_y/R$
(`tglf_eigensolver.f90:1039`), at 0.45–0.5 puts $\gamma$ (1.99× → ≈1.0×), $\omega$ (0.60× → ≈1.0×) and the eigenfunction on GS2 together (qp5.47).
Here the same knob on the 738 GS2-KBMs of the four `kbm_free` cubes. Settings are the qp5.44 baseline exactly (WIDTH 0.4, `FIND_WIDTH` F,
SAT_RULE 2, FILTER $-1.5$, NXGRID 32, IBRANCH $-1$, MHD rule off, $A_\parallel$ and $B_\parallel$ on); only GHAT changes.
GHAT 1 (the default) is the existing baseline leaf `fx_fm1p5_w0p4000_nb{16,24}`; GHAT 0.5 is run at NBASIS 16 and 24, and
0.4, 0.45, 0.6, 0.75 at NBASIS 24 (leaves `gh<GHAT>_nb<N>`, registry family `tglf_ghat_kbm_free`).
Everything is advisory (agent analysis).

In [ ]:
import os
import re
from multiprocessing import get_context
from pathlib import Path
import warnings
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import LogLocator, NullFormatter
from pyrokinetics import PyroHypercube
from pyrokinetics.diagnostics.extent import Extent

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file() and (path / "notebooks").is_dir())
paper_style = [ROOT / "src/general_analysis/paper.mplstyle"] + [p for p in (ROOT.parent / "styles/paper.mplstyle",) if p.exists()]
load_dotenv(ROOT / "local.env", override=True)
warnings.filterwarnings("ignore")
data_root = Path(os.path.expandvars(os.environ["GK_DATA_ROOT"]))     # the WORK tree
plot_dir = ROOT / "Plots" / "TGLF_GFTM_KBM_PAPER" / "tglf_ghat_database"
plot_dir.mkdir(parents=True, exist_ok=True)
n_cpu = int(os.environ.get("SLURM_CPUS_PER_TASK", 4))

cases = ["SPR-045", "M1", "R4", "R3"]
case_colour = {"SPR-045": "tab:blue", "M1": "tab:orange", "R4": "tab:green", "R3": "tab:red"}
ghat_nb24 = [0.0, 0.05, 0.1, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.6, 0.75, 1.0]
low, high = 1e-3, 1e1   # axis limits of every gamma-gamma figure; points outside are cut off, not removed from the data


def leaf_path(case, ghat, nbasis):
    name = f"fx_fm1p5_w0p4000_nb{nbasis}" if ghat == 1.0 else f"gh{ghat:g}_nb{nbasis}".replace(".", "p")
    return data_root / "TGLF/Runs/LATIN_HYPERCUBE" / case / "kbm_free" / name


def extent_95(squared_field):
    # pyro's 95 % extent, sample by sample; GS2's theta grid is NaN outside each sample's own points, so NaNs are dropped first
    return np.array([float(Extent._compute(row.dropna("theta").expand_dims("sample"), 0.95)[0][0]) if row.notnull().any() else np.nan
                     for row in squared_field.transpose("sample", "theta")])

## GS2 reference for the 738 KBMs

In [ ]:
gs2_rows = []
for case in cases:
    leaf = data_root / "GS2/Runs/LATIN_HYPERCUBE" / case / "kbm_free"
    gs2_scan = PyroHypercube(pyroscan_json=leaf / "pyroscan.json", load_base_pyro=True)
    gs2_scan.load_gk_output(netcdf_file=leaf / "pyroscan.nc")
    population = [str(name) for name in PyroHypercube(pyroscan_json=leaf_path(case, 1.0, 24) / "pyroscan.json", load_base_pyro=True).pyro_dict]
    gs2_output = gs2_scan.gk_output.data.set_index(sample="sample_name").sel(sample=population)
    gs2_rows.append(pd.DataFrame(dict(case=case, sample=population,
                                      gamma_gs2=gs2_output.growth_rate.pint.dequantify().values.ravel(),
                                      omega_gs2=gs2_output.mode_frequency.pint.dequantify().values.ravel(),
                                      gamma_units_gs2=re.sub(r"_pyroscan_base\d+", "", str(gs2_output.growth_rate.data.units)),
                                      phi_extent_gs2=extent_95(gs2_output.phi_squared.pint.dequantify().squeeze("kx")),
                                      apar_extent_gs2=extent_95(gs2_output.apar_squared.pint.dequantify().squeeze("kx")))))
gs2 = pd.concat(gs2_rows, ignore_index=True)
assert len(gs2) == 738, len(gs2)

## TGLF runs: dominant mode of every leaf
IBRANCH $-1$ returns the modes ordered by growth rate, so mode 0 is the dominant one (asserted on every leaf). A KBM-like mode is
*found* if $\gamma>0$ and $\omega>0$. The leaves `tk_tt0p1_nb24` and `tk_tt3p5355_nb24` of qp5.44 (THETA_TRAPPED 0.1 and the full geometric
trapped fraction) are also read, to define the $f_t$-insensitive modes.

In [ ]:
def load_leaf(spec):
    case, ghat, nbasis, leaf_dir = spec
    scan = PyroHypercube(pyroscan_json=leaf_dir / "pyroscan.json", load_base_pyro=True)
    scan.load_gk_output(netcdf_file=leaf_dir / "pyroscan.nc")
    output = scan.gk_output.data.set_index(sample="sample_name")
    growth_rate = output.growth_rate.pint.dequantify()
    assert (growth_rate.fillna(-np.inf).argmax("mode") == 0).all()
    dominant = output.isel(mode=0)  # mode0-guard-allow: IBRANCH -1, argmax == 0 asserted above
    return pd.DataFrame(dict(case=case, ghat=ghat, nbasis=nbasis, sample=[str(name) for name in output.sample.values],
                             gamma_units=re.sub(r"_pyroscan_base\d+", "", str(dominant.growth_rate.data.units)),
                             gamma=dominant.growth_rate.pint.dequantify().values,
                             omega=dominant.mode_frequency.pint.dequantify().values,
                             phi_extent=extent_95(abs(dominant.phi.pint.dequantify().squeeze("kx")) ** 2),
                             apar_extent=extent_95(abs(dominant.apar.pint.dequantify().squeeze("kx")) ** 2)))


specs = [(case, ghat, nbasis, leaf_path(case, ghat, nbasis)) for case in cases
         for ghat, nbasis in [(g, 24) for g in ghat_nb24] + [(0.5, 16), (1.0, 16)]]
ft_dir = lambda case, tag: data_root / "TGLF/Runs/LATIN_HYPERCUBE" / case / "kbm_free" / f"tk_{tag}_nb24"
specs += [(case, tag, 24, ft_dir(case, tag)) for case in cases for tag in ("tt0p1", "tt3p5355")]
with get_context("fork").Pool(n_cpu) as pool:
    loaded = pd.concat(pool.map(load_leaf, specs), ignore_index=True)
ft_runs = loaded[loaded.ghat.isin(["tt0p1", "tt3p5355"])]
runs = loaded[~loaded.ghat.isin(["tt0p1", "tt3p5355"])].astype({"ghat": float})
runs = runs.merge(gs2, on=["case", "sample"], validate="many_to_one")
assert (runs.groupby(["ghat", "nbasis"]).size() == 738).all()
assert (runs.gamma_units == runs.gamma_units_gs2).all(), "TGLF and GS2 growth rates are not in the same normalisation"
runs["found"] = (runs.gamma > 0) & (runs.omega > 0)
for quantity in ("gamma", "omega", "phi_extent", "apar_extent"):
    runs[f"{quantity}_ratio"] = (runs[quantity] / runs[f"{quantity}_gs2"]).where(runs.found)
print(runs.groupby(["ghat", "nbasis"]).found.mean().unstack().round(3).to_string())

## Step 1: per GHAT, pooled and per database (NBASIS 24)
$\gamma$ ratio $=\gamma_{\rm TGLF}/\gamma_{\rm GS2}$ etc.; medians over the found modes. "within $x$" $=\lvert\gamma_{\rm TGLF}/\gamma_{\rm GS2}-1\rvert\le x$ over all 738 of the group
(a mode that is not found counts as a miss).

In [ ]:
def metrics(group):
    return pd.Series({"n": len(group), "found": group.found.mean(), "median gamma ratio": group.gamma_ratio.median(),
                      "median omega ratio": group.omega_ratio.median(), "median phi extent ratio": group.phi_extent_ratio.median(),
                      "median apar extent ratio": group.apar_extent_ratio.median(),
                      "within 10 %": (abs(group.gamma_ratio - 1) <= 0.1).sum() / len(group),
                      "within 20 %": (abs(group.gamma_ratio - 1) <= 0.2).sum() / len(group)})


top = runs[runs.nbasis == 24]
summary = pd.concat([top.assign(case="pooled"), top]).groupby(["case", "ghat"]).apply(metrics)
summary.round(3).to_csv(plot_dir / "summary_by_ghat_nb24.csv")
print(summary.round(3).to_string())

Best GHAT per database, by the fraction within 20 % and by the median $\gamma$ ratio closest to 1 (log distance):

In [ ]:
best = summary.reset_index().groupby("case").apply(lambda group: pd.Series({
    "GHAT, most within 20 %": group.loc[group["within 20 %"].idxmax(), "ghat"],
    "GHAT, median gamma ratio nearest 1": group.loc[abs(np.log(group["median gamma ratio"])).idxmin(), "ghat"],
    "GHAT, median omega ratio nearest 1": group.loc[abs(np.log(group["median omega ratio"])).idxmin(), "ghat"]}))
best.to_csv(plot_dir / "best_ghat_per_database.csv")
print(best.to_string())

## NBASIS 16 against 24 (converged = $\lvert\gamma_{16}/\gamma_{24}-1\rvert\le5\,\%$, a stated default), GHAT 1 and 0.5

In [ ]:
pair = runs[runs.ghat.isin([1.0, 0.5])].pivot_table(index=["ghat", "case", "sample"], columns="nbasis", values=["gamma", "found"])
pair["change"] = abs(pair[("gamma", 16)] / pair[("gamma", 24)] - 1)
convergence = pd.concat([pair.reset_index().assign(case="pooled"), pair.reset_index()]).groupby(["case", "ghat"]).apply(lambda group: pd.Series({
    "within 5 %": (group["change"] <= 0.05).mean(), "median |gamma_16/gamma_24 - 1|": group["change"].median()}))
convergence.round(3).to_csv(plot_dir / "convergence_nb16_nb24.csv")
print(convergence.round(3).to_string())

## Figure 0: plain pooled $\gamma_{\rm TGLF}$ against $\gamma_{\rm GS2}$, coloured by database, every GHAT (NBASIS 24)
Plotting lines of the pooled `gamma` panels of `tglf_sat1_sat2` (`fig_sat*_gamma_vs_gs2_pooled`): log axes $10^{-3}$–$10^{1}$, $y=x$, case colours, $n$ per case in the legend.

In [ ]:
for ghat in ghat_nb24:
    arm = top[(top.ghat == ghat) & top.found]
    fig, ax = plt.subplots(figsize=(5.5, 5.5), constrained_layout=True)
    for case in cases:
        mask = (arm.case == case).values
        ax.scatter(arm.gamma_gs2[mask], arm.gamma[mask], s=14, alpha=0.7, color=case_colour[case], label=f"{case} (n={mask.sum()})")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.plot([low, high], [low, high], "k--", lw=1); ax.set_xlim(low, high); ax.set_ylim(low, high); ax.set_aspect("equal")
    ax.set_xlabel(r"$\gamma_{\rm GS2}$"); ax.set_ylabel(r"$\gamma_{\rm TGLF}$")
    ax.set_title(f"TGLF GHAT {ghat:g}, NBASIS 24, WIDTH 0.4 (median ratio {arm.gamma_ratio.median():.2f})", fontsize=11)
    ax.legend(loc="upper left")
    stem = f"fig_tglf_ghat{ghat:g}_gamma_vs_gs2_pooled".replace(".", "p")
    fig.savefig(plot_dir / f"{stem}.pdf"); fig.savefig(plot_dir / f"{stem}.png", dpi=150)
    plt.show()

## Figure 1: $\gamma_{\rm TGLF}$ against $\gamma_{\rm GS2}$, pooled, every GHAT (and GHAT 1 against 0.5)
Plotting code of `tglf_sat1_sat2` (cell 13), the overlay variable being GHAT instead of NBASIS. Solid lines: median $\gamma_{\rm TGLF}$ in log-spaced bins of $\gamma_{\rm GS2}$.

In [ ]:
bin_edges, min_per_bin = np.logspace(-2, 0.5, 11), 10   # binned-median lines; bins with fewer points are skipped
ghat_cmap = LinearSegmentedColormap.from_list("blue_red", ["tab:blue", "tab:red"])
# every GHAT (blue = 1 to red = 0), and the GHAT 1 against 0.5 pair
for suffix, overlay, point_alpha in [("", sorted(top.ghat.unique(), reverse=True), 0.08), ("_1_vs_0p5", [1.0, 0.5], 0.2)]:
    ghat_colours = ghat_cmap(np.linspace(0, 1, len(overlay)))

    with plt.style.context(paper_style):
        fig, ax = plt.subplots(figsize=(0.85 * 6.30, 0.6 * 6.30 * 0.85), layout="constrained")   # constrained: makes room for the outside legend
        for ghat, colour in zip(overlay, ghat_colours):
            arm = top[(top.ghat == ghat) & top.found]
            x, y = arm.gamma_gs2.values, arm.gamma.values
            print(f"GHAT {ghat:g}: {((x < low) | (y < low) | (x > high) | (y > high)).sum()} points outside axes, "
                  f"median gamma_T/gamma_G = {np.median(y / x):.2f}")
            ax.scatter(x, y, s=10, alpha=point_alpha, color=colour, lw=0)
            which = np.digitize(x, bin_edges)
            centres, medians = zip(*[(np.sqrt(bin_edges[i - 1] * bin_edges[i]), np.median(y[which == i]))
                                     for i in range(1, len(bin_edges)) if (which == i).sum() >= min_per_bin])
            ax.plot(centres, medians, "-o", color=colour, lw=1.5, ms=3, mec="k", mew=0.4,
                    label=rf"{ghat:g}: $\times${np.median(y / x):.1f}")
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.plot([low, high], [low, high], "k--", lw=0.8)
        ax.set_xlim(low, high); ax.set_ylim(low, high); ax.set_aspect("equal")
        for axis in (ax.xaxis, ax.yaxis): axis.set_major_locator(LogLocator(numticks=10))   # a label at every decade
        ax.xaxis.set_minor_locator(LogLocator(subs=np.arange(2, 10), numticks=10)); ax.xaxis.set_minor_formatter(NullFormatter())
        ax.tick_params(axis="x", which="minor", length=2, width=0.5)
        ax.set_xlabel(r"$\gamma_{\rm GS2}$"); ax.set_ylabel(r"$\gamma_{\rm TGLF}$")
        fig.legend(title="GHAT\n" + r"(median $\gamma_{\rm TGLF}/\gamma_{\rm GS2}$)", loc="outside right center")
        fig.savefig(plot_dir / f"fig_gamma_vs_gs2_pooled_ghat_overlay{suffix}.pdf"); fig.savefig(plot_dir / f"fig_gamma_vs_gs2_pooled_ghat_overlay{suffix}.png", dpi=150)
        plt.show()

## Figure 1b: mode frequency, $\omega_{\rm TGLF}$ against $\omega_{\rm GS2}$, pooled, every GHAT
Same layout as Figure 1 with $\omega$ (found modes have $\omega>0$, the ion direction) and every GHAT overlaid, blue (GHAT 1) to red (lowest). Lines: median $\omega_{\rm TGLF}$ in log-spaced bins of $\omega_{\rm GS2}$.

In [ ]:
found_modes = top[top.found]
omega_low, omega_high = 1e-3, 1e1   # as the gamma figures; a few near-zero TGLF frequencies lie below, counted here and kept in the data
print(((found_modes.omega < omega_low) | (found_modes.omega_gs2 < omega_low) | (found_modes.omega > omega_high) | (found_modes.omega_gs2 > omega_high)).groupby(found_modes.ghat).sum().to_dict(), "points outside the axes, per GHAT")
omega_bins = np.logspace(np.log10(omega_low), np.log10(omega_high), 11)
omega_overlay = sorted(top.ghat.unique(), reverse=True)
omega_colours = ghat_cmap(np.linspace(0, 1, len(omega_overlay)))

with plt.style.context(paper_style):
    fig, ax = plt.subplots(figsize=(0.85 * 6.30, 0.6 * 6.30 * 0.85), layout="constrained")
    for ghat, colour in zip(omega_overlay, omega_colours):
        arm = found_modes[found_modes.ghat == ghat]
        x, y = arm.omega_gs2.values, arm.omega.values
        ax.scatter(x, y, s=6, alpha=0.1, color=colour, lw=0)
        which = np.digitize(x, omega_bins)
        centres, medians = zip(*[(np.sqrt(omega_bins[i - 1] * omega_bins[i]), np.median(y[which == i]))
                                 for i in range(1, len(omega_bins)) if (which == i).sum() >= min_per_bin])
        ax.plot(centres, medians, "-o", color=colour, lw=1.5, ms=3, mec="k", mew=0.4, label=rf"{ghat:g}: $\times${np.median(y / x):.2f}")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.plot([omega_low, omega_high], [omega_low, omega_high], "k--", lw=0.8)
    ax.set_xlim(omega_low, omega_high); ax.set_ylim(omega_low, omega_high); ax.set_aspect("equal")
    ax.set_xlabel(r"$\omega_{\rm GS2}$"); ax.set_ylabel(r"$\omega_{\rm TGLF}$")
    fig.legend(title="GHAT\n" + r"(median $\omega_{\rm TGLF}/\omega_{\rm GS2}$)", loc="outside right center")
    fig.savefig(plot_dir / "fig_omega_vs_gs2_pooled_ghat_overlay.pdf"); fig.savefig(plot_dir / "fig_omega_vs_gs2_pooled_ghat_overlay.png", dpi=150)
    plt.show()

## Figure 1c: movement of the centre of mass with decreasing GHAT
The centre of mass of the cloud $(\log_{10}x_{\rm GS2},\ \log_{10}y_{\rm TGLF})$, $x,y=\gamma$ or $\omega$, at each GHAT, joined from GHAT 1 (blue) down to 0 (red); pooled in black-edged markers, each
database as a thin line. Only the modes found at **every** GHAT are used, so the GS2 side is the same at every GHAT and only TGLF moves
(a mode that appears or disappears between GHATs would otherwise shift the centre by selection). The cloud at GHAT 1 is shown faintly for context.

In [ ]:
found_by_ghat = top.pivot_table(index=["case", "sample"], columns="ghat", values="found", aggfunc="first")
common = found_by_ghat.index[found_by_ghat.all(axis=1)]
common_modes = top.set_index(["case", "sample"]).loc[common].reset_index()
print(len(common), "modes found at every GHAT, of 738")
centroid_rows = []
with plt.style.context(paper_style):
    fig, axes = plt.subplots(1, 2, figsize=(11, 5.2), constrained_layout=True)
    for ax, (reference, tglf_column, label) in zip(axes, [("gamma_gs2", "gamma", r"\gamma"), ("omega_gs2", "omega", r"\omega")]):
        cloud = common_modes[common_modes.ghat == 1.0]
        ax.scatter(cloud[reference], cloud[tglf_column], s=6, alpha=0.15, color="tab:blue", lw=0)
        for group in [*cases, "pooled"]:
            members = common_modes if group == "pooled" else common_modes[common_modes.case == group]
            path = members.groupby("ghat").apply(lambda modes: pd.Series({"x": 10 ** np.log10(modes[reference]).mean(), "y": 10 ** np.log10(modes[tglf_column]).mean()})).sort_index(ascending=False)
            centroid_rows += [dict(quantity=label.strip("\\"), group=group, ghat=ghat, x_gs2=row.x, y_tglf=row.y, ratio=row.y / row.x) for ghat, row in path.iterrows()]
            if group == "pooled":
                ax.plot(path.x, path.y, "-", color="k", lw=1.2, zorder=3)
                scatter = ax.scatter(path.x, path.y, c=path.index, cmap="coolwarm_r", vmin=0, vmax=1, s=60, edgecolor="k", zorder=4)
                for ghat in (1.0, 0.5, 0.0):
                    ax.annotate(f"GHAT {ghat:g}", (path.x[ghat], path.y[ghat]), textcoords="offset points", xytext=(10, -3), fontsize=9)
            else:
                ax.plot(path.x, path.y, "-", color=case_colour[group], lw=0.9, alpha=0.9, label=group)
        ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(5e-2, 3); ax.set_ylim(5e-2, 3); ax.set_aspect("equal")   # zoomed: the centres lie between 0.15 and 1
        ax.plot([5e-2, 3], [5e-2, 3], "k--", lw=0.8)
        ax.set_xlabel(rf"${label}_{{\rm GS2}}$"); ax.set_ylabel(rf"${label}_{{\rm TGLF}}$")
    axes[0].legend(title="database (lines)", loc="upper left", fontsize=8)
    fig.colorbar(scatter, ax=axes, shrink=0.6, label="GHAT (pooled centre of mass)")
    fig.savefig(plot_dir / "fig_centre_of_mass_vs_ghat.pdf"); fig.savefig(plot_dir / "fig_centre_of_mass_vs_ghat.png", dpi=150)
    plt.show()
pd.DataFrame(centroid_rows).round(4).to_csv(plot_dir / "centre_of_mass_by_ghat.csv", index=False)
print(pd.DataFrame(centroid_rows).query("group == 'pooled'").round(3).to_string(index=False))

## Figure 2: every GHAT, coloured by $\log_{10}$ of the $\phi$ extent ratio
Plotting code of `tglf_sat1_sat2` (cell 11), single panel.

In [ ]:
for ghat in ghat_nb24:
    arm = top[(top.ghat == ghat) & top.found]
    x, y = arm.gamma_gs2.values, arm.gamma.values
    log_ratio = np.log10(arm.phi_extent_ratio.values)
    print(f"GHAT {ghat:g}: {((x < low) | (y < low) | (x > high) | (y > high)).sum()} points outside axes")

    with plt.style.context(paper_style):
        fig, ax = plt.subplots(figsize=(0.6 * 6.30, 0.6 * 6.30 * 0.85))
        sc = ax.scatter(x, y, s=10, alpha=0.8, c=log_ratio, vmin=-1, vmax=1, cmap="coolwarm")
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.plot([low, high], [low, high], "k--", lw=0.8)
        ax.set_xlim(low, high); ax.set_ylim(low, high); ax.set_aspect("equal")
        ax.set_xlabel(r"$\gamma_{\rm GS2}$"); ax.set_ylabel(r"$\gamma_{\rm TGLF}$")
        fig.colorbar(sc, ax=ax, label=r"$\log_{10}(\theta_{\rm TGLF}/\theta_{\rm GS2})$")
        stem = f"fig_gamma_vs_gs2_ghat{ghat:g}_colour_phi_extent_ratio_pooled".replace(".", "p")
        fig.savefig(plot_dir / f"{stem}.pdf"); fig.savefig(plot_dir / f"{stem}.png", dpi=150)
        plt.show()

## Figure 3: within 10 / 20 % of GS2 and the median $\gamma$ ratio against GHAT, one line per database (NBASIS 24)

In [ ]:
with plt.style.context(paper_style):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), constrained_layout=True)
    for ax, column, label in zip(axes, ["within 10 %", "within 20 %", "median gamma ratio"],
                                 [r"fraction within 10 % of GS2", r"fraction within 20 % of GS2", r"median $\gamma_{\rm TGLF}/\gamma_{\rm GS2}$"]):
        for case in [*cases, "pooled"]:
            line = summary.loc[case, column]
            ax.plot(line.index, line.values, "-o", ms=3, color="k" if case == "pooled" else case_colour[case], lw=2.5 if case == "pooled" else 1.2, label=case)
        ax.set_xscale("symlog", linthresh=0.05); ax.set_xticks(ghat_nb24); ax.set_xticklabels([f"{g:g}" for g in ghat_nb24]); ax.minorticks_off()
        ax.set_xlabel("GHAT"); ax.set_ylabel(label)
    axes[2].set_yscale("log"); axes[2].axhline(1, color="grey", lw=0.8); axes[0].legend()
    fig.savefig(plot_dir / "fig_within_and_median_vs_ghat.pdf")
    plt.show()

## Does GHAT fix the 427 over-predicted, $f_t$-insensitive modes of qp5.44?
Over-predicted: baseline (GHAT 1) $\gamma$ ratio $>2$. $f_t$-insensitive: $\gamma(\mathrm{THETA\_TRAPPED}\ 3.5355)/\gamma(0.1)\le1.5$ at NBASIS 24 (qp5.44 defaults).

In [ ]:
sensitivity = ft_runs.pivot_table(index=["case", "sample"], columns="ghat", values="gamma")
sensitivity = (sensitivity["tt3p5355"] / sensitivity["tt0p1"]).rename("ft_sensitivity")
baseline = top[top.ghat == 1.0].set_index(["case", "sample"]).join(sensitivity)
baseline["class"] = np.where(baseline.gamma_ratio > 2, np.where(baseline.ft_sensitivity <= 1.5, "over-predicted, f_t-insensitive", "over-predicted, f_t-sensitive"), "ratio <= 2 or not found")
print(baseline["class"].value_counts().to_string())
classed = top.merge(baseline["class"].reset_index(), on=["case", "sample"])
by_class = classed.groupby(["class", "ghat"]).apply(metrics)
by_class.round(3).to_csv(plot_dir / "summary_by_class.csv")
print(by_class.round(3).to_string())

Per database for the insensitive class: median $\gamma$ and $\omega$ ratio and fraction within 20 % at GHAT 1 and 0.5.

In [ ]:
insensitive = classed[(classed["class"] == "over-predicted, f_t-insensitive") & classed.ghat.isin([1.0, 0.5])]
insensitive_table = insensitive.groupby(["case", "ghat"]).apply(metrics).round(3)
insensitive_table.to_csv(plot_dir / "insensitive_class_by_database.csv")
print(insensitive_table.to_string())

The 21 example modes of qp5.44 are not reused; `sample_0818` is checked below against qp5.47 (GHAT 0.5 should be $\gamma\approx1.0\times$, $\omega\approx1.0\times$).

In [ ]:
print(runs[(runs.case == "SPR-045") & (runs["sample"] == "sample_0818") & (runs.nbasis == 24)][["ghat", "gamma_ratio", "omega_ratio", "phi_extent_ratio", "apar_extent_ratio"]].round(3).to_string())